# ML Transfer Learning Pipeline: Han Pairwise Dataset

This notebook trains Logistic Regression and Random Forest models on `han_pairwise_dataset.csv` using the same feature set as the in-domain ML notebook:

- Character n-gram TF-IDF features
- Hand-crafted numeric similarity features
- Embedding cosine similarity feature

Split strategy:

- Train: 80%
- Development: 10%  
  - Saved for CNN/BERT comparison, but not used for ML training
- Test: 10%

The trained models are saved for later transfer-learning evaluation on `pairwise_gold_dataset.csv`.


In [1]:
!pip install pandas scikit-learn sentence-transformers

In [2]:
import os
import re
import json
import joblib
import pandas as pd
import numpy as np

from difflib import SequenceMatcher
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    precision_recall_fscore_support,
    accuracy_score,
    roc_auc_score
)
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix

from sentence_transformers import SentenceTransformer

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = "han_pairwise_dataset.csv"
OUTPUT_DIR = Path("ml_transfer_outputs")
MODEL_DIR = OUTPUT_DIR / "saved_models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Han pairwise dataset

In [3]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Label distribution:")
print(df["Label"].value_counts(dropna=False))

df.head()


Shape: (905364, 3)
Columns: ['OrgName_1', 'OrgName_2', 'Label']
Label distribution:
Label
0    715163
1    190201
Name: count, dtype: int64


,OrgName_1,OrgName_2,Label
0,CNPqConselho Nacional de Desenvolvimento Cient...,Conselho Nacional de Pesquisa,1
1,CNPqConselho Nacional de Desenvolvimento Cient...,CNPq,1
2,CNPqConselho Nacional de Desenvolvimento Cient...,National Council for Scientific and Technologi...,1
3,CNPqConselho Nacional de Desenvolvimento Cient...,Conselho Nacional de Pesquisa,1
4,CNPqConselho Nacional de Desenvolvimento Cient...,Conselho Nacional de Desenvolvimento Científic...,1


## 2. Basic cleaning and validation

In [4]:
required_cols = ["OrgName_1", "OrgName_2", "Label"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Keep only required columns for modeling
df = df[required_cols].copy()

# Remove rows with missing organization names or labels
df = df.dropna(subset=["OrgName_1", "OrgName_2", "Label"]).reset_index(drop=True)

# Ensure text columns are strings
df["OrgName_1"] = df["OrgName_1"].astype(str)
df["OrgName_2"] = df["OrgName_2"].astype(str)

# Ensure Label is integer type
df["Label"] = df["Label"].astype(int)

print("Cleaned shape:", df.shape)
print(df["Label"].value_counts())


Cleaned shape: (905357, 3)
Label
0    715157
1    190200
Name: count, dtype: int64


## 3. Train / Development / Test split: 8:1:1

The development set is saved for CNN/BERT experiments but is not used for ML model training.

In [5]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["Label"],
    random_state=RANDOM_SEED
)

dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["Label"],
    random_state=RANDOM_SEED
)

train_df = train_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape, train_df["Label"].value_counts().to_dict())
print("Development:", dev_df.shape, dev_df["Label"].value_counts().to_dict())
print("Test:", test_df.shape, test_df["Label"].value_counts().to_dict())

train_df.to_csv(OUTPUT_DIR / "han_train_80.csv", index=False, encoding="utf-8-sig")
dev_df.to_csv(OUTPUT_DIR / "han_development_10.csv", index=False, encoding="utf-8-sig")
test_df.to_csv(OUTPUT_DIR / "han_test_10.csv", index=False, encoding="utf-8-sig")


Train: (724285, 3) {0: 572125, 1: 152160}
Development: (90536, 3) {0: 71516, 1: 19020}
Test: (90536, 3) {0: 71516, 1: 19020}


## 4. Hand-crafted numeric similarity features

In [6]:
def normalize_text(text):
    # Convert text to lowercase and remove punctuation
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    # Tokenize normalized text
    return normalize_text(text).split()


def acronym(text):
    # Create acronym from organization name tokens
    tokens = tokenize(text)
    return "".join(t[0] for t in tokens if t)


def jaccard_similarity(text1, text2):
    # Compute token-level Jaccard similarity
    s1 = set(tokenize(text1))
    s2 = set(tokenize(text2))
    
    if len(s1) == 0 and len(s2) == 0:
        return 0
    
    return len(s1 & s2) / len(s1 | s2)


def sequence_similarity(text1, text2):
    # Compute character-level sequence similarity
    return SequenceMatcher(None, normalize_text(text1), normalize_text(text2)).ratio()


def acronym_match(text1, text2):
    # Check whether acronyms match
    a1 = acronym(text1)
    a2 = acronym(text2)
    
    if len(a1) < 2 or len(a2) < 2:
        return 0
    
    return int(a1 == a2)


def length_features(text1, text2):
    # Compute simple length-based features
    n1 = len(normalize_text(text1))
    n2 = len(normalize_text(text2))
    
    return {
        "len_1": n1,
        "len_2": n2,
        "len_diff": abs(n1 - n2),
        "len_ratio": min(n1, n2) / max(n1, n2) if max(n1, n2) > 0 else 0
    }


def build_numeric_features(dataframe):
    # Build hand-crafted numeric features
    rows = []
    
    for _, row in dataframe.iterrows():
        name1 = row["OrgName_1"]
        name2 = row["OrgName_2"]
        
        features = {
            "jaccard": jaccard_similarity(name1, name2),
            "sequence_similarity": sequence_similarity(name1, name2),
            "acronym_match": acronym_match(name1, name2)
        }
        
        features.update(length_features(name1, name2))
        rows.append(features)
    
    return pd.DataFrame(rows)


## 5. Embedding cosine similarity feature

This keeps the same embedding model used in the in-domain ML notebook.

In [7]:
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Encode all unique organization names from the Han dataset once
unique_names = pd.unique(
    pd.concat([df["OrgName_1"], df["OrgName_2"]], ignore_index=True)
)

print("Number of unique organization names:", len(unique_names))

name_embeddings = embedding_model.encode(
    unique_names.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

embedding_dict = {
    name: emb for name, emb in zip(unique_names, name_embeddings)
}


def get_embedding_cosine_from_dict(name1, name2, emb_dict):
    # Compute cosine similarity between two precomputed normalized embeddings
    emb1 = emb_dict[name1]
    emb2 = emb_dict[name2]
    return float(np.dot(emb1, emb2))


def add_embedding_cosine(dataframe, emb_dict):
    # Add embedding cosine similarity to a dataframe
    dataframe = dataframe.copy()
    dataframe["embedding_cosine"] = dataframe.apply(
        lambda row: get_embedding_cosine_from_dict(row["OrgName_1"], row["OrgName_2"], emb_dict),
        axis=1
    )
    return dataframe

train_df = add_embedding_cosine(train_df, embedding_dict)
dev_df = add_embedding_cosine(dev_df, embedding_dict)
test_df = add_embedding_cosine(test_df, embedding_dict)

train_df.head()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6156.13it/s]


Number of unique organization names: 215819


Batches: 100%|██████████| 3373/3373 [01:04<00:00, 52.66it/s]


,OrgName_1,OrgName_2,Label,embedding_cosine
0,Institut Ocelových Konstrukcí (Czechia),University of South Bohemia [158/2016/P],0,0.427468
1,IMTM,IMMD,0,0.810522
2,Autoliv (Germany),Bundesverband CarSharing,0,0.479652
3,SpazioDati (Italy),Stam (Italy),0,0.692387
4,SBBG,SBGHC,0,0.863025


## 6. Feature builder

The feature set is aligned with the in-domain ML notebook:

1. Character n-gram TF-IDF from paired organization names  
2. Hand-crafted numeric similarity features  
3. Embedding cosine similarity  


In [8]:
def make_pair_text(dataframe):
    # Combine two organization names as pair text for character n-gram features
    return (
        dataframe["OrgName_1"].astype(str)
        + " [SEP] "
        + dataframe["OrgName_2"].astype(str)
    )


def build_train_test_features(train_dataframe, test_dataframe):
    # Build TF-IDF features using only the training set
    train_pair_text = make_pair_text(train_dataframe)
    test_pair_text = make_pair_text(test_dataframe)
    
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 5),
        min_df=1
    )
    
    X_train_ngram = vectorizer.fit_transform(train_pair_text)
    X_test_ngram = vectorizer.transform(test_pair_text)
    
    # Build numeric features
    train_numeric = build_numeric_features(train_dataframe)
    test_numeric = build_numeric_features(test_dataframe)
    
    # Add embedding cosine similarity
    train_numeric["embedding_cosine"] = train_dataframe["embedding_cosine"].values
    test_numeric["embedding_cosine"] = test_dataframe["embedding_cosine"].values
    
    numeric_feature_names = train_numeric.columns.tolist()
    
    X_train_numeric = csr_matrix(train_numeric.values)
    X_test_numeric = csr_matrix(test_numeric.values)
    
    # Combine n-gram and numeric features
    X_train = hstack([X_train_ngram, X_train_numeric])
    X_test = hstack([X_test_ngram, X_test_numeric])
    
    y_train = train_dataframe["Label"].values
    y_test = test_dataframe["Label"].values
    
    return X_train, X_test, y_train, y_test, vectorizer, numeric_feature_names


X_train, X_test, y_train, y_test, vectorizer, numeric_feature_names = build_train_test_features(
    train_df,
    test_df
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Numeric features:", numeric_feature_names)


X_train shape: (724285, 801552)
X_test shape: (90536, 801552)
Numeric features: ['jaccard', 'sequence_similarity', 'acronym_match', 'len_1', 'len_2', 'len_diff', 'len_ratio', 'embedding_cosine']


## 7. Train Logistic Regression and Random Forest

In [9]:
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_SEED
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        max_depth=50,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1
    )
}

trained_models = {}

for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)
    trained_models[model_name] = model
    print(f"Done: {model_name}")


Training LogisticRegression...


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Done: LogisticRegression
Training RandomForest...
Done: RandomForest


## 8. Evaluate models on the Han test set

In [10]:
def evaluate_models(models_dict, X_test, y_test, test_dataframe):
    # Store model-level results
    model_results = []
    
    # Store pair-level prediction results
    pair_predictions = []
    
    for model_name, model in models_dict.items():
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        
        overall_precision, overall_recall, overall_f1, _ = precision_recall_fscore_support(
            y_test,
            y_pred,
            average="binary",
            pos_label=1,
            zero_division=0
        )
        
        class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
            y_test,
            y_pred,
            labels=[0, 1],
            average=None,
            zero_division=0
        )
        
        auc = roc_auc_score(y_test, y_prob)
        acc = accuracy_score(y_test, y_pred)
        
        model_results.append({
            "Model": model_name,
            "Accuracy": acc,
            "ROC_AUC": auc,
            
            "Match_Precision": class_precision[1],
            "Match_Recall": class_recall[1],
            "Match_F1": class_f1[1],
            "Match_Support": class_support[1],
            
            "NonMatch_Precision": class_precision[0],
            "NonMatch_Recall": class_recall[0],
            "NonMatch_F1": class_f1[0],
            "NonMatch_Support": class_support[0],
            
            "Overall_PositiveClass_Precision": overall_precision,
            "Overall_PositiveClass_Recall": overall_recall,
            "Overall_PositiveClass_F1": overall_f1
        })
        
        temp_pred_df = test_dataframe.copy()
        temp_pred_df["Model"] = model_name
        temp_pred_df["TrueLabel"] = y_test
        temp_pred_df["PredictedLabel"] = y_pred
        temp_pred_df["Probability_Match"] = y_prob
        temp_pred_df["Probability_NonMatch"] = 1 - y_prob
        
        pair_predictions.append(temp_pred_df)
        
        print(f"===== {model_name} =====")
        print(classification_report(
            y_test,
            y_pred,
            target_names=["NonMatch", "Match"],
            digits=4,
            zero_division=0
        ))
    
    model_results_df = pd.DataFrame(model_results)
    pair_predictions_df = pd.concat(pair_predictions, ignore_index=True)
    
    return model_results_df, pair_predictions_df


model_results_df, pair_predictions_df = evaluate_models(
    trained_models,
    X_test,
    y_test,
    test_df
)

model_results_df


===== LogisticRegression =====
              precision    recall  f1-score   support

    NonMatch     0.9478    0.8790    0.9121     71516
       Match     0.6427    0.8179    0.7198     19020

    accuracy                         0.8662     90536
   macro avg     0.7952    0.8485    0.8160     90536
weighted avg     0.8837    0.8662    0.8717     90536

===== RandomForest =====
              precision    recall  f1-score   support

    NonMatch     0.9537    0.8741    0.9121     71516
       Match     0.6396    0.8403    0.7263     19020

    accuracy                         0.8670     90536
   macro avg     0.7966    0.8572    0.8192     90536
weighted avg     0.8877    0.8670    0.8731     90536



,Model,Accuracy,ROC_AUC,Match_Precision,Match_Recall,Match_F1,Match_Support,NonMatch_Precision,NonMatch_Recall,NonMatch_F1,NonMatch_Support,Overall_PositiveClass_Precision,Overall_PositiveClass_Recall,Overall_PositiveClass_F1
0,LogisticRegression,0.866208,0.918190,0.642665,0.817928,0.719782,19020,0.947791,0.879048,0.912126,71516,0.642665,0.817928,0.719782
1,RandomForest,0.866981,0.930802,0.639613,0.840273,0.726339,19020,0.953653,0.874084,0.912137,71516,0.639613,0.840273,0.726339


## 9. Save ML result files

In [11]:
# Save pair-level prediction results
pair_predictions_df.to_csv(
    OUTPUT_DIR / "ml_pair_level_predictions_with_probability.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save model-level summary
summary_df = model_results_df.copy()
summary_df.to_csv(
    OUTPUT_DIR / "ml_summary_match_nonmatch.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(OUTPUT_DIR / "ml_pair_level_predictions_with_probability.csv")
print(OUTPUT_DIR / "ml_summary_match_nonmatch.csv")


Saved:
ml_transfer_outputs/ml_pair_level_predictions_with_probability.csv
ml_transfer_outputs/ml_summary_match_nonmatch.csv


## 10. Save trained models for transfer-learning evaluation

Each saved artifact includes:

- trained classifier
- fitted TF-IDF vectorizer
- numeric feature names
- embedding model name
- random seed and metadata

The SentenceTransformer model itself is not saved inside the `.joblib` file. Instead, its model name is saved and reloaded when applying the classifier to `pairwise_gold_dataset.csv`.

In [12]:
for model_name, model in trained_models.items():
    artifact = {
        "model_name": model_name,
        "classifier": model,
        "vectorizer": vectorizer,
        "numeric_feature_names": numeric_feature_names,
        "embedding_model_name": EMBEDDING_MODEL_NAME,
        "random_seed": RANDOM_SEED,
        "feature_description": {
            "text_features": "char_wb TF-IDF, ngram_range=(2, 5), min_df=1",
            "numeric_features": numeric_feature_names,
            "embedding_feature": "embedding_cosine from sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
        }
    }
    
    output_path = MODEL_DIR / f"{model_name}_han_transfer_model.joblib"
    joblib.dump(artifact, output_path)
    print("Saved:", output_path)


Saved: ml_transfer_outputs/saved_models/LogisticRegression_han_transfer_model.joblib
Saved: ml_transfer_outputs/saved_models/RandomForest_han_transfer_model.joblib


## 11. Function for applying saved ML models to `pairwise_gold_dataset.csv`

Run this section later when you want to evaluate the Han-trained ML models on the gold dataset.

In [13]:
def build_features_with_existing_vectorizer(dataframe, vectorizer, embedding_model_name):
    # Build TF-IDF features using the saved vectorizer
    pair_text = make_pair_text(dataframe)
    X_ngram = vectorizer.transform(pair_text)
    
    # Build numeric features
    numeric = build_numeric_features(dataframe)
    
    # Compute embedding cosine similarity for the new dataset
    emb_model = SentenceTransformer(embedding_model_name)
    
    unique_names = pd.unique(
        pd.concat(
            [dataframe["OrgName_1"], dataframe["OrgName_2"]],
            ignore_index=True
        )
    )
    
    embeddings = emb_model.encode(
        unique_names.tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    
    emb_dict = {
        name: emb for name, emb in zip(unique_names, embeddings)
    }
    
    dataframe = add_embedding_cosine(dataframe, emb_dict)
    numeric["embedding_cosine"] = dataframe["embedding_cosine"].values
    
    X_numeric = csr_matrix(numeric.values)
    
    # Combine TF-IDF and numeric features
    X = hstack([X_ngram, X_numeric])
    
    return X, dataframe

In [14]:
def evaluate_saved_model_on_gold(
    gold_path,
    artifact_path,
    output_prediction_path=None,
    output_summary_path=None
):
    # Load gold dataset
    gold_df = pd.read_csv(gold_path)
    
    required_cols = ["OrgName_1", "OrgName_2", "Label"]
    missing_cols = [col for col in required_cols if col not in gold_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in gold dataset: {missing_cols}")
    
    gold_df = gold_df[required_cols].dropna().copy().reset_index(drop=True)
    gold_df["OrgName_1"] = gold_df["OrgName_1"].astype(str)
    gold_df["OrgName_2"] = gold_df["OrgName_2"].astype(str)
    gold_df["Label"] = gold_df["Label"].astype(int)
    
    # Load saved artifact
    artifact = joblib.load(artifact_path)
    model = artifact["classifier"]
    vectorizer = artifact["vectorizer"]
    embedding_model_name = artifact["embedding_model_name"]
    model_name = artifact["model_name"]
    
    # Build features
    X_gold, gold_df = build_features_with_existing_vectorizer(
        gold_df,
        vectorizer,
        embedding_model_name
    )
    
    y_true = gold_df["Label"].values
    y_pred = model.predict(X_gold)
    y_prob = model.predict_proba(X_gold)[:, 1]
    
    print(f"\n===== Gold evaluation: {model_name} =====")
    
    print(classification_report(
        y_true,
        y_pred,
        target_names=["NonMatch", "Match"],
        digits=4,
        zero_division=0
    ))
    
    # Create summary table: Precision, Recall, F1, Support
    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=["NonMatch", "Match"],
        output_dict=True,
        zero_division=0
    )
    
    summary_df = pd.DataFrame(report_dict).T.reset_index()
    summary_df = summary_df.rename(columns={
        "index": "Class",
        "precision": "Precision",
        "recall": "Recall",
        "f1-score": "F1",
        "support": "Support"
    })
    
    summary_df.insert(0, "Model", model_name)
    summary_df["Accuracy"] = accuracy_score(y_true, y_pred)
    summary_df["ROC_AUC"] = roc_auc_score(y_true, y_prob)
    
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("ROC_AUC:", roc_auc_score(y_true, y_prob))
    
    # Save prediction-level results
    pred_df = gold_df.copy()
    pred_df["Model"] = model_name
    pred_df["TrueLabel"] = y_true
    pred_df["PredictedLabel"] = y_pred
    pred_df["Probability_Match"] = y_prob
    pred_df["Probability_NonMatch"] = 1 - y_prob
    
    if output_prediction_path is not None:
        pred_df.to_csv(output_prediction_path, index=False, encoding="utf-8-sig")
        print("Saved predictions to:", output_prediction_path)
    
    if output_summary_path is not None:
        summary_df.to_csv(output_summary_path, index=False, encoding="utf-8-sig")
        print("Saved summary to:", output_summary_path)
    
    return pred_df, summary_df


### Example: evaluate saved models on `pairwise_gold_dataset.csv`

Uncomment and run this section later when the gold dataset is ready in the same column format.

In [15]:
lr_gold_pred, lr_gold_summary = evaluate_saved_model_on_gold(
    gold_path="pairwise_gold_dataset.csv",
    artifact_path="ml_transfer_outputs/saved_models/LogisticRegression_han_transfer_model.joblib",
    output_prediction_path="lr_gold_predictions_with_probability.csv",
    output_summary_path="lr_gold_summary_match_nonmatch.csv"
)

rf_gold_pred, rf_gold_summary = evaluate_saved_model_on_gold(
    gold_path="pairwise_gold_dataset.csv",
    artifact_path="ml_transfer_outputs/saved_models/RandomForest_han_transfer_model.joblib",
    output_prediction_path="rf_gold_predictions_with_probability.csv",
    output_summary_path="rf_gold_summary_match_nonmatch.csv"
)


Batches: 100%|██████████| 20/20 [00:00<00:00, 61.19it/s]



===== Gold evaluation: LogisticRegression =====
              precision    recall  f1-score   support

    NonMatch     0.8726    0.5905    0.7043      2830
       Match     0.2174    0.5689    0.3146       566

    accuracy                         0.5869      3396
   macro avg     0.5450    0.5797    0.5095      3396
weighted avg     0.7634    0.5869    0.6394      3396

Accuracy: 0.5868669022379269
ROC_AUC: 0.6090742798636517
Saved predictions to: lr_gold_predictions_with_probability.csv
Saved summary to: lr_gold_summary_match_nonmatch.csv


Batches: 100%|██████████| 20/20 [00:00<00:00, 61.21it/s]



===== Gold evaluation: RandomForest =====
              precision    recall  f1-score   support

    NonMatch     0.9125    0.6378    0.7508      2830
       Match     0.2772    0.6943    0.3962       566

    accuracy                         0.6472      3396
   macro avg     0.5948    0.6661    0.5735      3396
weighted avg     0.8066    0.6472    0.6917      3396

Accuracy: 0.6472320376914017
ROC_AUC: 0.7017068511281198
Saved predictions to: rf_gold_predictions_with_probability.csv
Saved summary to: rf_gold_summary_match_nonmatch.csv


In [16]:
ml_gold_summary = pd.concat(
    [lr_gold_summary, rf_gold_summary],
    ignore_index=True
)

ml_gold_summary.to_csv(
    "ml_gold_summary_match_nonmatch.csv",
    index=False,
    encoding="utf-8-sig"
)